# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [1]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 2.3.1
   - numpy version: 2.2.6


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [ ]:
# Set up AWS credentials for the session
os.environ["AWS_ACCESS_KEY_ID"] = "YOUR_AWS_ACCESS_KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_AWS_SECRET_ACCESS_KEY"
os.environ["AWS_SESSION_TOKEN"] = "YOUR_AWS_SESSION_TOKEN"

In [3]:
import os, boto3
print("key prefix:", str(os.environ.get("AWS_ACCESS_KEY_ID"))[:8])
sts = boto3.client("sts", region_name=os.environ.get("AWS_REGION", "us-east-1"))
print(sts.get_caller_identity())

key prefix: ASIAXKOM
{'UserId': 'AROAXKOM7JUY2D7C6ZZWT:user5314219=8d697da3-a889-4060-bca4-959aa392b382', 'Account': '503477914929', 'Arn': 'arn:aws:sts::503477914929:assumed-role/voclabs/user5314219=8d697da3-a889-4060-bca4-959aa392b382', 'ResponseMetadata': {'RequestId': 'dae82856-a8a3-417e-9561-c0b183668be5', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'dae82856-a8a3-417e-9561-c0b183668be5', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzg5NzEzODAyOTkxOlI6RGxZWFJPOWs=', 'content-type': 'text/xml', 'content-length': '510', 'date': 'Fri, 18 Sep 2026 06:43:23 GMT'}, 'RetryAttempts': 0}}


In [4]:
def _first_existing(*paths):
    for p in paths:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return paths[0]

BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS = _first_existing(
    os.path.join(DATA_DIR, "ecom_orders_postgres.csv"),
    os.path.join(BASE_DIR, "ecom_orders_postgres.csv"),
    "/home/workdir/attachments/ecom_orders_postgres.csv",
)
CSV_EVENTS = _first_existing(
    os.path.join(DATA_DIR, "ecom_events_cassandra.csv"),
    os.path.join(BASE_DIR, "ecom_events_cassandra.csv"),
    "/home/workdir/attachments/ecom_events_cassandra.csv",
)
CSV_EDGES = _first_existing(
    os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv"),
    os.path.join(BASE_DIR, "ecom_graph_edges_neo4j.csv"),
    "/home/workdir/attachments/ecom_graph_edges_neo4j.csv",
)
DDL_MD_PATH = _first_existing(
    os.path.join(BASE_DIR, "project-ddl-long.md"),
    "/home/workdir/attachments/project-ddl-long.md",
)
MERMAID_MD = _first_existing(
    os.path.join(BASE_DIR, "project-mermaid-diagram.md"),
    "/home/workdir/attachments/project-mermaid-diagram.md",
)
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "500"))

PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

print("Configuration loaded")
print("  CSV orders", CSV_ORDERS, os.path.isfile(CSV_ORDERS))
print("  CSV events", CSV_EVENTS, os.path.isfile(CSV_EVENTS))
print("  CSV edges ", CSV_EDGES, os.path.isfile(CSV_EDGES))
print("  DDL", DDL_MD_PATH, os.path.isfile(DDL_MD_PATH))
print("  Postgres", f"{PG_HOST}:{PG_PORT}/{PG_DB}")
print("  Redshift", REDSHIFT_DATABASE, REDSHIFT_WORKGROUP)
if AWS_ACCESS_KEY_ID and not str(AWS_ACCESS_KEY_ID).startswith("YOUR_"):
    print("  AWS credentials found", AWS_ACCESS_KEY_ID[:8], "...")
else:
    print("  WARNING: set Cloud Resources keys in the previous cell before Task 4")


Configuration loaded
  CSV orders /workspace/data/ecom_orders_postgres.csv True
  CSV events /workspace/data/ecom_events_cassandra.csv True
  CSV edges  /workspace/data/ecom_graph_edges_neo4j.csv True
  DDL /workspace/project-ddl-long.md True
  Postgres localhost:5432/postgres
  Redshift ecom udacity-dwh-wg
  AWS credentials found ASIAXKOM ...


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.


In [5]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [6]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [7]:
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()
print("="*60)
print("TASK 1: Data Exploration")
print("="*60)
print("ORDERS DATA (PostgreSQL)")
print("   Shape:", orders_df.shape)
print("   Columns:", list(orders_df.columns))
display(orders_df.head(3))
print(orders_df.dtypes)


Reading CSV files...
TASK 1: Data Exploration
ORDERS DATA (PostgreSQL)
   Shape: (2500, 26)
   Columns: ['order_id', 'customer_id', 'order_datetime', 'ship_datetime', 'channel', 'device_type', 'browser', 'country', 'state', 'payment_method', 'campaign', 'primary_category', 'num_distinct_items', 'subtotal_usd', 'discount_rate', 'discount_amount_usd', 'shipping_method', 'shipping_cost_usd', 'tax_rate', 'tax_amount_usd', 'order_total_usd', 'order_weight_kg', 'delivery_days', 'on_time_delivery', 'authorization_approved', 'returned']


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False


order_id                   object
customer_id                object
order_datetime             object
ship_datetime              object
channel                    object
device_type                object
browser                    object
country                    object
state                      object
payment_method             object
campaign                   object
primary_category           object
num_distinct_items          int64
subtotal_usd              float64
discount_rate             float64
discount_amount_usd       float64
shipping_method            object
shipping_cost_usd         float64
tax_rate                  float64
tax_amount_usd            float64
order_total_usd           float64
order_weight_kg           float64
delivery_days               int64
on_time_delivery             bool
authorization_approved       bool
returned                     bool
dtype: object


In [8]:
print("EVENTS DATA (Cassandra)")
print("   Shape:", events_df.shape)
print("   Columns:", list(events_df.columns))
display(events_df.head(3))
print(events_df["event_type"].value_counts())


EVENTS DATA (Cassandra)
   Shape: (2500, 24)
   Columns: ['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200000,C47857,S7645111197,page_view,2024-11-14 00:17:09,desktop,Firefox,Linux,organic_search,IN,...,331,35,18.12,0.165,0.000,NaN,3,NaN,NaN,NONE
1,EVT200001,C83195,S1423573902,page_view,2024-08-15 10:42:18,mobile,Chrome,Linux,social,DE,...,432,26,33.62,0.083,0.262,NaN,2,NaN,NaN,NONE
2,EVT200002,C27664,S2829853742,product_view,2024-05-03 23:28:24,mobile,Chrome,Android,email,BR,...,360,6,12.26,0.237,0.162,NaN,5,P5735,Home,FREESHIP


event_type
product_view        696
page_view           641
add_to_cart         480
checkout_start      259
payment_attempt     218
purchase            155
return_initiated     51
Name: count, dtype: int64


In [9]:
print("GRAPH EDGES DATA (Neo4j)")
print("   Shape:", edges_df.shape)
print("   Columns:", list(edges_df.columns))
display(edges_df.head(3))
print(edges_df["relationship"].value_counts())
print(edges_df.groupby(["from_node_type","to_node_type"]).size())


GRAPH EDGES DATA (Neo4j)
   Shape: (2500, 23)
   Columns: ['edge_id', 'from_node_id', 'from_node_type', 'to_node_id', 'to_node_type', 'relationship', 'timestamp', 'order_id', 'category', 'customer_segment', 'edge_strength', 'price_bucket', 'region', 'state', 'campaign', 'same_household', 'prior_interactions', 'dwell_seconds', 'product_id', 'unit_price_usd', 'quantity', 'returned_flag', 'auth_approved']


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False


relationship
VIEWED              830
PURCHASED           469
ADDED_TO_CART       441
ALSO_BOUGHT_WITH    393
REFERRED_FRIEND     256
RETURNS             111
Name: count, dtype: int64
from_node_type  to_node_type
Customer        Customer         256
                Product         1851
Product         Product          393
dtype: int64


In [10]:
print("="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)
print("Primary Keys:")
print("  orders.order_id unique", orders_df["order_id"].nunique(), "/", len(orders_df))
print("  events.event_id unique", events_df["event_id"].nunique(), "/", len(events_df))
print("  edges.edge_id unique", edges_df["edge_id"].nunique(), "/", len(edges_df))
print()
print("Foreign Key Relationships:")
print("  customer_id joins orders, events, and Customer-typed graph nodes")
print("  product_id joins events and Product-typed graph nodes")
print("  order_id joins orders and PURCHASED/RETURNS graph edges")
print()
print("Date/Time Fields -> dim_date:")
print("  orders: order_datetime, ship_datetime")
print("  events: event_ts")
print("  edges: timestamp")
print()
print("Categorical Fields (potential dimensions):")
for name, df, cols in [
    ("orders", orders_df, ["channel","device_type","browser","country","payment_method","campaign","shipping_method","primary_category"]),
    ("events", events_df, ["event_type","device_type","browser","os","referrer","country","ab_variant","promo_code"]),
    ("edges", edges_df, ["relationship","category","customer_segment","price_bucket","region","campaign"]),
]:
    print(" ", name)
    for c in cols:
        print(f"    {c:22} {df[c].nunique(dropna=True)}")


KEY FIELDS AND RELATIONSHIPS
Primary Keys:
  orders.order_id unique 2500 / 2500
  events.event_id unique 2500 / 2500
  edges.edge_id unique 2500 / 2500

Foreign Key Relationships:
  customer_id joins orders, events, and Customer-typed graph nodes
  product_id joins events and Product-typed graph nodes
  order_id joins orders and PURCHASED/RETURNS graph edges

Date/Time Fields -> dim_date:
  orders: order_datetime, ship_datetime
  events: event_ts
  edges: timestamp

Categorical Fields (potential dimensions):
  orders
    channel                5
    device_type            3
    browser                5
    country                6
    payment_method         5
    campaign               6
    shipping_method        3
    primary_category       10
  events
    event_type             7
    device_type            3
    browser                5
    os                     5
    referrer               6
    country                9
    ab_variant             2
    promo_code             5
  e

In [11]:
print("="*60)
print("FIELD MAPPING AND DATA QUALITY")
print("="*60)
mapping = pd.DataFrame([
    ["PostgreSQL","order_id","dw_fact_orders.order_id","degenerate natural key"],
    ["PostgreSQL","customer_id","dw_dim_customer.customer_id -> customer_sk","SK lookup"],
    ["PostgreSQL","order_datetime","dw_dim_date as order_date_key","YYYYMMDD"],
    ["PostgreSQL","ship_datetime","dw_dim_date as ship_date_key","YYYYMMDD"],
    ["PostgreSQL","channel","dw_dim_channel.channel_sk","SK lookup"],
    ["PostgreSQL","device_type","dw_dim_device.device_sk","SK lookup"],
    ["PostgreSQL","browser","dw_dim_browser.browser_sk","SK lookup"],
    ["PostgreSQL","payment_method","dw_dim_payment_method.payment_method_sk","SK lookup"],
    ["PostgreSQL","shipping_method","dw_dim_shipping_method.shipping_method_sk","SK lookup"],
    ["PostgreSQL","campaign","dw_dim_campaign.campaign_sk","SK lookup"],
    ["PostgreSQL","money / qty / flags","dw_fact_orders measures","additive facts"],
    ["Cassandra","event_id","dw_fact_events.event_id","degenerate NK"],
    ["Cassandra","customer_id","dw_dim_customer -> customer_sk","SK lookup"],
    ["Cassandra","product_id","dw_dim_product -> product_sk","SK lookup (nullable on page_view)"],
    ["Cassandra","event_ts","dw_dim_date as event_date_key","YYYYMMDD"],
    ["Cassandra","os / referrer / ab_variant","dim_os / dim_referrer / dim_ab_variant","SK lookup"],
    ["Cassandra","cart_value, dwell, fraud","dw_fact_events measures","measures"],
    ["Neo4j","edge_id","dw_fact_graph_edges.edge_id","degenerate NK"],
    ["Neo4j","from_node_id + from_node_type","from_customer_sk or from_product_sk","typed SK lookup"],
    ["Neo4j","to_node_id + to_node_type","to_customer_sk or to_product_sk","typed SK lookup"],
    ["Neo4j","relationship","dw_fact_graph_edges.relationship","degenerate attribute"],
    ["Neo4j","edge_strength, qty, price","dw_fact_graph_edges measures","measures"],
], columns=["source_system","source_field","warehouse_target","conformance"])
display(mapping)
print("ID strategy: strip whitespace; empty string -> NULL; keep C/P/ORD/EVT/EDGE prefixes; do not recode UK->GB.")
print("Nulls on keys:")
print("orders", orders_df[["order_id","customer_id"]].isna().sum().to_dict())
print("events", events_df[["event_id","customer_id","product_id"]].isna().sum().to_dict())
print("edges", edges_df[["edge_id","from_node_id","to_node_id","product_id","order_id"]].isna().sum().to_dict())


FIELD MAPPING AND DATA QUALITY


,source_system,source_field,warehouse_target,conformance
0,PostgreSQL,order_id,dw_fact_orders.order_id,degenerate natural key
1,PostgreSQL,customer_id,dw_dim_customer.customer_id -> customer_sk,SK lookup
2,PostgreSQL,order_datetime,dw_dim_date as order_date_key,YYYYMMDD
3,PostgreSQL,ship_datetime,dw_dim_date as ship_date_key,YYYYMMDD
4,PostgreSQL,channel,dw_dim_channel.channel_sk,SK lookup
5,PostgreSQL,device_type,dw_dim_device.device_sk,SK lookup
6,PostgreSQL,browser,dw_dim_browser.browser_sk,SK lookup
7,PostgreSQL,payment_method,dw_dim_payment_method.payment_method_sk,SK lookup
8,PostgreSQL,shipping_method,dw_dim_shipping_method.shipping_method_sk,SK lookup
9,PostgreSQL,campaign,dw_dim_campaign.campaign_sk,SK lookup


ID strategy: strip whitespace; empty string -> NULL; keep C/P/ORD/EVT/EDGE prefixes; do not recode UK->GB.
Nulls on keys:
orders {'order_id': 0, 'customer_id': 0}
events {'event_id': 0, 'customer_id': 0, 'product_id': 1118}
edges {'edge_id': 0, 'from_node_id': 0, 'to_node_id': 0, 'product_id': 0, 'order_id': 1527}


---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [12]:
print("="*60)
print("TASK 2: Warehouse Schema Design")
print("="*60)
print("Reading DDL file:", DDL_MD_PATH)
with open(DDL_MD_PATH) as f:
    ddl_content = f.read()
blocks = re.findall(r"```sql(.*?)```", ddl_content, flags=re.DOTALL|re.IGNORECASE)
all_sql = "\n".join(blocks)
tables = re.findall(r"CREATE TABLE\s+([\w.]+)", all_sql, flags=re.IGNORECASE)
views = re.findall(r"CREATE OR REPLACE VIEW\s+([\w.]+)", all_sql, flags=re.IGNORECASE)
print("Staging + dim + fact tables:", len(tables))
for t in tables:
    print(" ", t)
print("Views:", views)
print("Staging tables:", sum(1 for t in tables if "stg." in t or t.endswith("_raw")))
print("Dimension tables:", sum(1 for t in tables if "dim_" in t))
print("Fact tables:", sum(1 for t in tables if "fact_" in t))


TASK 2: Warehouse Schema Design
Reading DDL file: /workspace/project-ddl-long.md
Staging + dim + fact tables: 18
  stg.orders_raw
  stg.events_raw
  stg.edges_raw
  dw.dim_date
  dw.dim_customer
  dw.dim_product
  dw.dim_campaign
  dw.dim_channel
  dw.dim_device
  dw.dim_browser
  dw.dim_os
  dw.dim_referrer
  dw.dim_shipping_method
  dw.dim_payment_method
  dw.dim_ab_variant
  dw.fact_orders
  dw.fact_events
  dw.fact_graph_edges
Views: ['dw.v_lookup_customer', 'dw.v_lookup_product']
Staging tables: 3
Dimension tables: 12
Fact tables: 3


In [13]:
print("="*60)
print("SCHEMA DESIGN DOCUMENTATION")
print("="*60)
schema_doc = """
## Staging Tables (stg schema -> public.stg_* in this lab)
stg.orders_raw, stg.events_raw, stg.edges_raw
Purpose: land raw extracts 1:1 with source CSVs. Permissive types. No DISTKEY required.

## Dimension Tables (dw schema -> public.dw_*)
dim_date: calendar attributes, date_key = YYYYMMDD, DISTSTYLE ALL, SORT date_key
dim_customer: customer_sk IDENTITY, NK customer_id, SCD2 columns, DISTKEY(customer_id)
dim_product: product_sk IDENTITY, NK product_id, SCD2 columns, DISTKEY(product_id)
Junk dims ALL: campaign, channel, device, browser, os, referrer, shipping, payment, ab_variant

## Fact Tables
fact_orders grain = one row per order_id
  DISTKEY(customer_sk) SORTKEY(order_date_key)
fact_events grain = one row per event_id
  DISTKEY(customer_sk) SORTKEY(event_date_key)
fact_graph_edges grain = one row per edge_id
  DISTKEY(to_product_sk) SORTKEY(event_date_key)

## Why these keys
Customer DIST collocates orders + clickstream for funnel/LTV.
Product DIST on graph edges collocates also-bought / viewed-with.
Date SORT prunes revenue and event time windows.
ALL on small dims avoids broadcast shuffle.
ENCODE zstd on VARCHAR/DECIMAL columns.
"""
print(schema_doc)


SCHEMA DESIGN DOCUMENTATION

## Staging Tables (stg schema -> public.stg_* in this lab)
stg.orders_raw, stg.events_raw, stg.edges_raw
Purpose: land raw extracts 1:1 with source CSVs. Permissive types. No DISTKEY required.

## Dimension Tables (dw schema -> public.dw_*)
dim_date: calendar attributes, date_key = YYYYMMDD, DISTSTYLE ALL, SORT date_key
dim_customer: customer_sk IDENTITY, NK customer_id, SCD2 columns, DISTKEY(customer_id)
dim_product: product_sk IDENTITY, NK product_id, SCD2 columns, DISTKEY(product_id)
Junk dims ALL: campaign, channel, device, browser, os, referrer, shipping, payment, ab_variant

## Fact Tables
fact_orders grain = one row per order_id
  DISTKEY(customer_sk) SORTKEY(order_date_key)
fact_events grain = one row per event_id
  DISTKEY(customer_sk) SORTKEY(event_date_key)
fact_graph_edges grain = one row per edge_id
  DISTKEY(to_product_sk) SORTKEY(event_date_key)

## Why these keys
Customer DIST collocates orders + clickstream for funnel/LTV.
Product DIST on g

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [14]:
# ========= PostgreSQL Functions =========

def pg_connect():
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PW)

def pg_load_orders(df: pd.DataFrame):
    conn = pg_connect()
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS src_orders")
    cur.execute(
        "CREATE TABLE src_orders ("
        "order_id TEXT PRIMARY KEY, customer_id TEXT,"
        "order_datetime TIMESTAMP, ship_datetime TIMESTAMP,"
        "channel TEXT, device_type TEXT, browser TEXT, country TEXT, state TEXT,"
        "payment_method TEXT, campaign TEXT, primary_category TEXT,"
        "num_distinct_items INT, subtotal_usd NUMERIC, discount_rate NUMERIC,"
        "discount_amount_usd NUMERIC, shipping_method TEXT, shipping_cost_usd NUMERIC,"
        "tax_rate NUMERIC, tax_amount_usd NUMERIC, order_total_usd NUMERIC,"
        "order_weight_kg NUMERIC, delivery_days INT,"
        "on_time_delivery BOOLEAN, authorization_approved BOOLEAN, returned BOOLEAN)"
    )
    cols = [c for c,_ in ORDERS_COLSPEC]
    rows = [tuple(None if pd.isna(r[c]) else r[c] for c in cols) for _, r in df.iterrows()]
    execute_values(cur, "INSERT INTO src_orders (" + ",".join(cols) + ") VALUES %s", rows, page_size=BATCH_SIZE)
    conn.commit(); cur.close(); conn.close()
    print("Postgres loaded", len(rows))

def extract_from_pg() -> pd.DataFrame:
    try:
        conn = pg_connect()
        df = pd.read_sql("SELECT * FROM src_orders", conn)
        conn.close()
        print("Extracted from Postgres", len(df))
        return trim_df(df)
    except Exception as e:
        print("Postgres fallback to CSV:", type(e).__name__, e)
        return orders_df.copy()

print("PostgreSQL functions defined")


PostgreSQL functions defined


In [18]:
# ========= Cassandra Functions =========

def cas_connect():
    auth = PlainTextAuthProvider(CAS_USER, CAS_PW) if CAS_USER else None
    cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth)
    session = cluster.connect()
    session.execute(
        "CREATE KEYSPACE IF NOT EXISTS " + CAS_KEYSPACE +
        " WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}"
    )
    session.set_keyspace(CAS_KEYSPACE)
    return cluster, session

def cas_load_events(df: pd.DataFrame):
    from cassandra.concurrent import execute_concurrent_with_args
    from datetime import datetime as dt

    cluster, session = cas_connect()
    session.execute("DROP TABLE IF EXISTS events")
    session.execute(
        "CREATE TABLE events (event_id text PRIMARY KEY, customer_id text, session_id text,"
        " event_type text, event_ts timestamp, device_type text, browser text, os text,"
        " referrer text, country text, state text, ab_variant text, is_logged_in boolean,"
        " page_depth int, latency_ms int, dwell_seconds int, cart_value_usd decimal,"
        " discount_rate decimal, fraud_score decimal, payment_outcome text, sequence_num int,"
        " product_id text, category text, promo_code text)"
    )
    cols = [c for c, _ in EVENTS_COLSPEC]
    insert = session.prepare(
        "INSERT INTO events (" + ",".join(cols) + ") VALUES (" + ",".join(["?"] * len(cols)) + ")"
    )

    def cell(col, val):
        if pd.isna(val):
            return None
        if col == "event_ts":
            ts = pd.to_datetime(val, errors="coerce")
            return None if pd.isna(ts) else ts.to_pydatetime()
        if col in ("is_logged_in",):
            if isinstance(val, str):
                return val.strip().lower() in ("true", "1", "t", "yes")
            return bool(val)
        if col in ("page_depth", "latency_ms", "dwell_seconds", "sequence_num"):
            return int(val)
        if col in ("cart_value_usd", "discount_rate", "fraud_score"):
            return float(val)
        return val

    args = [tuple(cell(c, r[c]) for c in cols) for _, r in df.iterrows()]
    execute_concurrent_with_args(session, insert, args, concurrency=32)
    cluster.shutdown()
    print("Cassandra loaded", len(args))
    
def extract_from_cas() -> pd.DataFrame:
    try:
        cluster, session = cas_connect()
        rows = list(session.execute("SELECT * FROM events"))
        cluster.shutdown()
        df = pd.DataFrame(rows)
        print("Extracted from Cassandra", len(df))
        return trim_df(df)
    except Exception as e:
        print("Cassandra fallback to CSV:", type(e).__name__, e)
        return events_df.copy()

print("Cassandra functions defined")


Cassandra functions defined


In [16]:
# ========= Neo4j Functions =========

def neo4j_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def neo4j_load_edges(df: pd.DataFrame):
    drv = neo4j_driver()
    recs = df.replace({np.nan: None}).to_dict("records")
    cypher = (
        "UNWIND $rows AS row "
        "MERGE (a {id: row.from_node_id}) "
        "SET a.node_type = row.from_node_type "
        "MERGE (b {id: row.to_node_id}) "
        "SET b.node_type = row.to_node_type "
        "MERGE (a)-[r:REL {edge_id: row.edge_id}]->(b) "
        "SET r += row"
    )
    with drv.session() as s:
        try:
            s.run("CREATE CONSTRAINT customer_id IF NOT EXISTS FOR (c:Customer) REQUIRE c.id IS UNIQUE")
        except Exception:
            pass
        s.run("MATCH (n) DETACH DELETE n")
        for i in range(0, len(recs), BATCH_SIZE):
            s.run(cypher, rows=recs[i:i+BATCH_SIZE])
    drv.close()
    print("Neo4j loaded", len(recs))

def extract_from_neo4j() -> pd.DataFrame:
    try:
        drv = neo4j_driver()
        with drv.session() as s:
            result = s.run(
                "MATCH (a)-[r:REL]->(b) "
                "RETURN r.edge_id AS edge_id, a.id AS from_node_id, r.from_node_type AS from_node_type, "
                "b.id AS to_node_id, r.to_node_type AS to_node_type, r.relationship AS relationship, "
                "r.timestamp AS timestamp, r.order_id AS order_id, r.category AS category, "
                "r.customer_segment AS customer_segment, r.edge_strength AS edge_strength, "
                "r.price_bucket AS price_bucket, r.region AS region, r.state AS state, "
                "r.campaign AS campaign, r.same_household AS same_household, "
                "r.prior_interactions AS prior_interactions, r.dwell_seconds AS dwell_seconds, "
                "r.product_id AS product_id, r.unit_price_usd AS unit_price_usd, "
                "r.quantity AS quantity, r.returned_flag AS returned_flag, r.auth_approved AS auth_approved"
            )
            df = pd.DataFrame([dict(rec) for rec in result])
        drv.close()
        print("Extracted from Neo4j", len(df))
        return trim_df(df)
    except Exception as e:
        print("Neo4j fallback to CSV:", type(e).__name__, e)
        return edges_df.copy()

print("Neo4j functions defined")


Neo4j functions defined


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [19]:
print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)
try:
    print("Loading orders into PostgreSQL...")
    pg_load_orders(orders_df)
except Exception as e:
    print("PostgreSQL load skipped:", type(e).__name__, e)
try:
    print("Loading events into Cassandra...")
    cas_load_events(events_df)
except Exception as e:
    print("Cassandra load skipped:", type(e).__name__, e)
try:
    print("Loading edges into Neo4j...")
    neo4j_load_edges(edges_df)
except Exception as e:
    print("Neo4j load skipped:", type(e).__name__, e)
print("Source-system step finished (live DB or skipped)")


TASK 3: Loading Source Systems
Loading orders into PostgreSQL...
Postgres loaded 2500
Loading events into Cassandra...
Cassandra loaded 2500
Loading edges into Neo4j...
Neo4j loaded 2500
Source-system step finished (live DB or skipped)


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [20]:
print("="*60)
print("Extracting from Source Systems")
print("="*60)
orders_extracted = extract_from_pg()
events_extracted = extract_from_cas()
edges_extracted  = extract_from_neo4j()
print("Extracted", len(orders_extracted), len(events_extracted), len(edges_extracted))

def conform(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype(str).str.strip()
            out.loc[out[c].isin(["nan","None",""]), c] = np.nan
    return out

orders = conform(orders_extracted, ["order_id","customer_id"])
events = conform(events_extracted, ["event_id","customer_id","session_id","product_id"])
edges  = conform(edges_extracted, ["edge_id","from_node_id","to_node_id","order_id","product_id"])

orders = orders[[c for c,_ in ORDERS_COLSPEC if c in orders.columns]].copy()
events = events[[c for c,_ in EVENTS_COLSPEC if c in events.columns]].copy()
edges  = edges[[c for c,_ in EDGES_COLSPEC if c in edges.columns]].copy()

for col in ["order_datetime","ship_datetime"]:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")
if "event_ts" in events.columns:
    events["event_ts"] = pd.to_datetime(events["event_ts"], errors="coerce")
if "timestamp" in edges.columns:
    edges["timestamp"] = pd.to_datetime(edges["timestamp"], errors="coerce")

print("Conformed staging frames:", orders.shape, events.shape, edges.shape)
display(orders.head(2))
display(events.head(2))
display(edges.head(2))
print("Task 3 complete")


Extracting from Source Systems
Extracted from Postgres 2500


/tmp/ipykernel_62/2613999082.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM src_orders", conn)


Extracted from Cassandra 2500
Extracted from Neo4j 2500
Extracted 2500 2500 2500
Conformed staging frames: (2500, 26) (2500, 24) (2500, 23)


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0,0.0,838.39,0.1,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0,0.0,388.60,0.2,3,True,True,False


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200166,C14954,S2042011579,product_view,2025-05-20 12:16:03,desktop,Firefox,iOS,direct,GB,...,432,32,8.03999999999999914734871708787977695465087890625,0.17799999999999999156230501284881029278039932...,0,NaN,7,P4704,Books,HOLIDAY20
1,EVT202405,C88709,S8197777931,checkout_start,2025-06-14 02:20:07,mobile,Opera,iOS,affiliate,FR,...,318,59,97.5100000000000051159076974727213382720947265625,0,0.25,NaN,1,NaN,NaN,WELCOME10


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE301000,C98639,Customer,C35650,Customer,REFERRED_FRIEND,2025-01-19 12:12:56,NaN,Books,Loyal,...,NJ,BackToSchool,False,4,13,P9026,22.55,1,False,False
1,EDGE301001,C65361,Customer,P3546,Product,RETURNS,2024-11-22 02:35:58,ORD100941,Automotive,New,...,ND,Loyalty,False,4,70,P3546,54.87,1,True,False


Task 3 complete


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [21]:
# ========= Redshift Functions =========
session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless WORKGROUP or provisioned CLUSTER+DB_USER")
    return base

def rs_exec(sql: str, return_results=False, timeout_s=900):
    resp = rsd.execute_statement(Sql=sql, **_rs_kwargs())
    sid = resp["Id"]
    t0 = time.time()
    while True:
        desc = rsd.describe_statement(Id=sid)
        st = desc["Status"]
        if st in ("FINISHED", "FAILED", "ABORTED"):
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError("Redshift statement timed out: " + sid)
        time.sleep(0.6)
    if st != "FINISHED":
        raise RuntimeError(desc.get("Error", str(desc)))
    if not return_results:
        return None
    out, token = [], None
    while True:
        kw = {"Id": sid}
        if token:
            kw["NextToken"] = token
        res = rsd.get_statement_result(**kw)
        cols = [c["name"] for c in res.get("ColumnMetadata", [])]
        for rec in res.get("Records", []):
            row = {}
            for name, cell in zip(cols, rec):
                row[name] = next(iter(cell.values())) if cell else None
            out.append(row)
        token = res.get("NextToken")
        if not token:
            break
    return out

def _sql_lit(val, kind):
    if val is None or pd.isna(val):
        return "NULL"
    if kind == "s":
        return "'" + str(val).replace("'", "''") + "'"
    if kind == "ts":
        return "'" + pd.to_datetime(val).strftime("%Y-%m-%d %H:%M:%S") + "'"
    if kind == "b":
        if isinstance(val, str):
            return "TRUE" if val.strip().lower() in ("true","1","t","yes") else "FALSE"
        return "TRUE" if bool(val) else "FALSE"
    if kind == "i":
        return str(int(val))
    return str(float(val))

def rs_batch_insert(table: str, colspec, df: pd.DataFrame):
    cols = [c for c,_ in colspec if c in df.columns]
    kinds = {c:k for c,k in colspec}
    iterator = range(0, len(df), BATCH_SIZE)
    if TQDM:
        iterator = tqdm(iterator, desc=table)
    n = 0
    for i in iterator:
        chunk = df.iloc[i:i+BATCH_SIZE]
        values = []
        for _, r in chunk.iterrows():
            values.append("(" + ",".join(_sql_lit(r[c], kinds[c]) for c in cols) + ")")
        sql = "INSERT INTO " + table + " (" + ",".join(cols) + ") VALUES " + ",".join(values)
        rs_exec(sql)
        n += len(chunk)
    print("inserted", n, "into", table)

print("Redshift functions defined")


Redshift functions defined


## Task 4.2: Execute DDL and Create Tables

In [22]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)
print("Step 1: Executing DDL")
with open(DDL_MD_PATH) as f:
    md_content = f.read()
blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL|re.IGNORECASE)

def rewrite_sql(s: str) -> str:
    s = re.sub(r"\bstg\.", "public.stg_", s)
    s = re.sub(r"\bdw\.", "public.dw_", s)
    s = re.sub(r"CREATE SCHEMA IF NOT EXISTS\s+\w+\s*;", "", s, flags=re.I)
    return s.strip()

statements = []
for b in blocks:
    cleaned = rewrite_sql(b)
    for part in re.split(r";\s*\n", cleaned):
        stmt = part.strip()
        if stmt and not stmt.startswith("--"):
            statements.append(stmt if stmt.endswith(";") else stmt + ";")

print("SQL statements parsed:", len(statements))
for i, stmt in enumerate(statements, 1):
    preview = stmt.replace("\n", " ")[:100]
    try:
        rs_exec(stmt)
        print(" OK", i, preview)
    except Exception as e:
        print(" ERR", i, preview, "->", e)


TASK 4: Loading Data into Redshift
Step 1: Executing DDL
SQL statements parsed: 20
 OK 1 CREATE TABLE public.stg_orders_raw (   order_id              VARCHAR(32),   customer_id           VA
 OK 2 CREATE TABLE public.stg_events_raw (   event_id          VARCHAR(32),   customer_id       VARCHAR(32
 OK 3 CREATE TABLE public.stg_edges_raw (   edge_id            VARCHAR(32),   from_node_id       VARCHAR(3
 OK 4 CREATE TABLE public.dw_dim_date (   date_key        INTEGER   NOT NULL,  -- yyyymmdd format   date_a
 OK 5 CREATE TABLE public.dw_dim_customer (   customer_sk      BIGINT IDENTITY(1,1),   customer_id      VA
 OK 6 CREATE TABLE public.dw_dim_product (   product_sk             BIGINT IDENTITY(1,1),   product_id    
 OK 7 CREATE TABLE public.dw_dim_campaign (   campaign_sk  BIGINT IDENTITY(1,1),   campaign     VARCHAR(32
 OK 8 CREATE TABLE public.dw_dim_channel (   channel_sk  BIGINT IDENTITY(1,1),   channel     VARCHAR(32) E
 OK 9 CREATE TABLE public.dw_dim_device (   device_sk   BIGIN

## Task 4.3: Load Staging Tables

In [23]:
print("Step 2: Loading staging tables")
print("  stg_orders_raw", len(orders))
rs_batch_insert("public.stg_orders_raw", ORDERS_COLSPEC, orders)
print("  stg_events_raw", len(events))
rs_batch_insert("public.stg_events_raw", EVENTS_COLSPEC, events)
print("  stg_edges_raw", len(edges))
rs_batch_insert("public.stg_edges_raw", EDGES_COLSPEC, edges)
print("Staging tables loaded")


Step 2: Loading staging tables
  stg_orders_raw 2500
inserted 2500 into public.stg_orders_raw
  stg_events_raw 2500
inserted 2500 into public.stg_events_raw
  stg_edges_raw 2500
inserted 2500 into public.stg_edges_raw
Staging tables loaded


## Task 4.4: Populate Dimension Tables

In [24]:
print("Step 3: Populating dimension tables")

rs_exec("DELETE FROM public.dw_dim_date;")
rs_exec("""
INSERT INTO public.dw_dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
SELECT DISTINCT
    CAST(to_char(dt, 'YYYYMMDD') AS INTEGER),
    dt,
    EXTRACT(YEAR FROM dt)::SMALLINT,
    EXTRACT(QUARTER FROM dt)::SMALLINT,
    EXTRACT(MONTH FROM dt)::SMALLINT,
    EXTRACT(DAY FROM dt)::SMALLINT,
    EXTRACT(WEEK FROM dt)::SMALLINT,
    EXTRACT(DOW FROM dt)::SMALLINT,
    (EXTRACT(DOW FROM dt) IN (0,6))::BOOLEAN
FROM (
    SELECT order_datetime::date AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
    UNION SELECT ship_datetime::date FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
    UNION SELECT event_ts::date FROM public.stg_events_raw WHERE event_ts IS NOT NULL
    UNION SELECT timestamp::date FROM public.stg_edges_raw WHERE timestamp IS NOT NULL
) dates
WHERE dt IS NOT NULL;
""")
print(" dim_date")

rs_exec("DELETE FROM public.dw_dim_customer;")
rs_exec("""
INSERT INTO public.dw_dim_customer (customer_id, country, state, customer_segment, is_logged_in, effective_from, effective_to, is_current)
SELECT customer_id, MAX(country), MAX(state), MAX(segment), BOOL_OR(logged), MIN(ts), TIMESTAMP '9999-12-31', TRUE
FROM (
    SELECT customer_id, country, state, CAST(NULL AS VARCHAR) AS segment, FALSE AS logged, order_datetime AS ts
    FROM public.stg_orders_raw
    UNION ALL
    SELECT customer_id, country, state, CAST(NULL AS VARCHAR), COALESCE(is_logged_in, FALSE), event_ts
    FROM public.stg_events_raw
    UNION ALL
    SELECT from_node_id, CAST(NULL AS VARCHAR), state, customer_segment, FALSE, timestamp
    FROM public.stg_edges_raw WHERE from_node_type = 'Customer'
) x
WHERE customer_id IS NOT NULL
GROUP BY customer_id;
""")
print(" dim_customer")

rs_exec("DELETE FROM public.dw_dim_product;")
rs_exec("""
INSERT INTO public.dw_dim_product (product_id, category, price_bucket, current_unit_price_usd, effective_from, effective_to, is_current)
SELECT product_id, MAX(category), MAX(price_bucket), MAX(price), MIN(ts), TIMESTAMP '9999-12-31', TRUE
FROM (
    SELECT product_id, category, CAST(NULL AS VARCHAR) AS price_bucket, CAST(NULL AS DECIMAL(12,2)) AS price, event_ts AS ts
    FROM public.stg_events_raw WHERE product_id IS NOT NULL
    UNION ALL
    SELECT product_id, category, price_bucket, unit_price_usd, timestamp
    FROM public.stg_edges_raw WHERE product_id IS NOT NULL
    UNION ALL
    SELECT to_node_id, category, price_bucket, unit_price_usd, timestamp
    FROM public.stg_edges_raw WHERE to_node_type = 'Product'
    UNION ALL
    SELECT from_node_id, category, price_bucket, unit_price_usd, timestamp
    FROM public.stg_edges_raw WHERE from_node_type = 'Product'
) p
WHERE product_id IS NOT NULL
GROUP BY product_id;
""")
print(" dim_product")

def seed_dim(table, col, sources):
    unions = " UNION ".join(["SELECT DISTINCT " + col + " AS v FROM " + s + " WHERE " + col + " IS NOT NULL" for s in sources])
    rs_exec("DELETE FROM " + table + ";")
    rs_exec("INSERT INTO " + table + " (" + col + ") SELECT v FROM (" + unions + ") s;")
    print(" ", table)

seed_dim("public.dw_dim_campaign", "campaign", ["public.stg_orders_raw","public.stg_edges_raw"])
seed_dim("public.dw_dim_channel", "channel", ["public.stg_orders_raw"])
seed_dim("public.dw_dim_device", "device_type", ["public.stg_orders_raw","public.stg_events_raw"])
seed_dim("public.dw_dim_browser", "browser", ["public.stg_orders_raw","public.stg_events_raw"])
seed_dim("public.dw_dim_os", "os", ["public.stg_events_raw"])
seed_dim("public.dw_dim_referrer", "referrer", ["public.stg_events_raw"])
seed_dim("public.dw_dim_shipping_method", "shipping_method", ["public.stg_orders_raw"])
seed_dim("public.dw_dim_payment_method", "payment_method", ["public.stg_orders_raw"])
seed_dim("public.dw_dim_ab_variant", "ab_variant", ["public.stg_events_raw"])
print("All dimension tables populated")


Step 3: Populating dimension tables
 dim_date
 dim_customer
 dim_product
  public.dw_dim_campaign
  public.dw_dim_channel
  public.dw_dim_device
  public.dw_dim_browser
  public.dw_dim_os
  public.dw_dim_referrer
  public.dw_dim_shipping_method
  public.dw_dim_payment_method
  public.dw_dim_ab_variant
All dimension tables populated


## Task 4.5: Populate Fact Tables

In [25]:
print("Step 4: Populating fact tables")
rs_exec("DELETE FROM public.dw_fact_orders;")
rs_exec("""
INSERT INTO public.dw_fact_orders (
  order_id, customer_sk, order_date_key, ship_date_key,
  channel_sk, device_sk, browser_sk, campaign_sk, payment_method_sk, shipping_method_sk,
  primary_category, num_distinct_items, subtotal_usd, discount_rate, discount_amount_usd,
  shipping_cost_usd, tax_rate, tax_amount_usd, order_total_usd, order_weight_kg,
  delivery_days, on_time_delivery, authorization_approved, returned
)
SELECT
  o.order_id, c.customer_sk,
  CAST(to_char(o.order_datetime::date,'YYYYMMDD') AS INTEGER),
  CAST(to_char(o.ship_datetime::date,'YYYYMMDD') AS INTEGER),
  ch.channel_sk, d.device_sk, b.browser_sk, camp.campaign_sk, pm.payment_method_sk, sm.shipping_method_sk,
  o.primary_category, o.num_distinct_items, o.subtotal_usd, o.discount_rate, o.discount_amount_usd,
  o.shipping_cost_usd, o.tax_rate, o.tax_amount_usd, o.order_total_usd, o.order_weight_kg,
  o.delivery_days, o.on_time_delivery, o.authorization_approved, o.returned
FROM public.stg_orders_raw o
LEFT JOIN public.dw_dim_customer c ON c.customer_id = o.customer_id AND c.is_current = TRUE
LEFT JOIN public.dw_dim_channel ch ON ch.channel = o.channel
LEFT JOIN public.dw_dim_device d ON d.device_type = o.device_type
LEFT JOIN public.dw_dim_browser b ON b.browser = o.browser
LEFT JOIN public.dw_dim_campaign camp ON camp.campaign = o.campaign
LEFT JOIN public.dw_dim_payment_method pm ON pm.payment_method = o.payment_method
LEFT JOIN public.dw_dim_shipping_method sm ON sm.shipping_method = o.shipping_method;
""")
print(" fact_orders")

rs_exec("DELETE FROM public.dw_fact_events;")
rs_exec("""
INSERT INTO public.dw_fact_events (
  event_id, customer_sk, product_sk, event_date_key, session_id, event_type,
  device_sk, browser_sk, os_sk, referrer_sk, ab_variant_sk,
  page_depth, latency_ms, dwell_seconds, cart_value_usd, discount_rate, fraud_score,
  payment_outcome, sequence_num, category, promo_code
)
SELECT
  e.event_id, c.customer_sk, p.product_sk,
  CAST(to_char(e.event_ts::date,'YYYYMMDD') AS INTEGER),
  e.session_id, e.event_type,
  d.device_sk, b.browser_sk, o.os_sk, r.referrer_sk, a.ab_variant_sk,
  e.page_depth, e.latency_ms, e.dwell_seconds, e.cart_value_usd, e.discount_rate, e.fraud_score,
  e.payment_outcome, e.sequence_num, e.category, e.promo_code
FROM public.stg_events_raw e
LEFT JOIN public.dw_dim_customer c ON c.customer_id = e.customer_id AND c.is_current = TRUE
LEFT JOIN public.dw_dim_product p ON p.product_id = e.product_id AND p.is_current = TRUE
LEFT JOIN public.dw_dim_device d ON d.device_type = e.device_type
LEFT JOIN public.dw_dim_browser b ON b.browser = e.browser
LEFT JOIN public.dw_dim_os o ON o.os = e.os
LEFT JOIN public.dw_dim_referrer r ON r.referrer = e.referrer
LEFT JOIN public.dw_dim_ab_variant a ON a.ab_variant = e.ab_variant;
""")
print(" fact_events")

rs_exec("DELETE FROM public.dw_fact_graph_edges;")
rs_exec("""
INSERT INTO public.dw_fact_graph_edges (
  edge_id, event_date_key, relationship,
  from_customer_sk, to_customer_sk, from_product_sk, to_product_sk,
  order_id, category, campaign_sk, customer_segment, region, state,
  edge_strength, price_bucket, prior_interactions, dwell_seconds,
  unit_price_usd, quantity, returned_flag, auth_approved
)
SELECT
  g.edge_id,
  CAST(to_char(g.timestamp::date,'YYYYMMDD') AS INTEGER),
  g.relationship,
  fc.customer_sk, tc.customer_sk, fp.product_sk, tp.product_sk,
  g.order_id, g.category, camp.campaign_sk, g.customer_segment, g.region, g.state,
  g.edge_strength, g.price_bucket, g.prior_interactions, g.dwell_seconds,
  g.unit_price_usd, g.quantity, g.returned_flag, g.auth_approved
FROM public.stg_edges_raw g
LEFT JOIN public.dw_dim_customer fc
  ON fc.customer_id = g.from_node_id AND g.from_node_type = 'Customer' AND fc.is_current = TRUE
LEFT JOIN public.dw_dim_customer tc
  ON tc.customer_id = g.to_node_id AND g.to_node_type = 'Customer' AND tc.is_current = TRUE
LEFT JOIN public.dw_dim_product fp
  ON fp.product_id = g.from_node_id AND g.from_node_type = 'Product' AND fp.is_current = TRUE
LEFT JOIN public.dw_dim_product tp
  ON tp.product_id = CASE WHEN g.to_node_type = 'Product' THEN g.to_node_id ELSE g.product_id END
 AND tp.is_current = TRUE
LEFT JOIN public.dw_dim_campaign camp ON camp.campaign = g.campaign;
""")
print(" fact_graph_edges")
print("Task 4 complete")


Step 4: Populating fact tables
 fact_orders
 fact_events
 fact_graph_edges
Task 4 complete


---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

In [26]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)
print("DISTKEY / SORTKEY justification")
dist_sort = pd.DataFrame([
    ["dw_fact_orders","DISTKEY(customer_sk)","Collocate with events for customer funnel and LTV"],
    ["dw_fact_orders","SORTKEY(order_date_key)","Range-restrict daily and monthly GMV queries"],
    ["dw_fact_events","DISTKEY(customer_sk)","Same customer slice as orders"],
    ["dw_fact_events","SORTKEY(event_date_key)","Time-window clickstream scans"],
    ["dw_fact_graph_edges","DISTKEY(to_product_sk)","Product affinity / ALSO_BOUGHT_WITH"],
    ["dw_fact_graph_edges","SORTKEY(event_date_key)","Time filter on relationships"],
    ["dw_dim_customer","DISTKEY(customer_id)","Collocate NK during ETL lookups"],
    ["dw_dim_product","DISTKEY(product_id)","Collocate NK during ETL lookups"],
    ["dw_dim_date and junk dims","DISTSTYLE ALL","Broadcast; tables are tiny"],
], columns=["table","key","justification"])
display(dist_sort)

print("Creating materialized view dw_mv_daily_revenue")
rs_exec("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
rs_exec(
    "CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS "
    "SELECT order_date_key, COUNT(*) AS orders, SUM(order_total_usd) AS revenue_usd, "
    "AVG(order_total_usd) AS avg_order_value, "
    "SUM(CASE WHEN returned THEN 1 ELSE 0 END) AS returned_orders, "
    "AVG(CASE WHEN on_time_delivery THEN 1.0 ELSE 0.0 END) AS on_time_rate "
    "FROM public.dw_fact_orders GROUP BY order_date_key;"
)
print("MV created. After future loads run: REFRESH MATERIALIZED VIEW public.dw_mv_daily_revenue;")


TASK 5: Optimize Performance
DISTKEY / SORTKEY justification


,table,key,justification
0,dw_fact_orders,DISTKEY(customer_sk),Collocate with events for customer funnel and LTV
1,dw_fact_orders,SORTKEY(order_date_key),Range-restrict daily and monthly GMV queries
2,dw_fact_events,DISTKEY(customer_sk),Same customer slice as orders
3,dw_fact_events,SORTKEY(event_date_key),Time-window clickstream scans
4,dw_fact_graph_edges,DISTKEY(to_product_sk),Product affinity / ALSO_BOUGHT_WITH
5,dw_fact_graph_edges,SORTKEY(event_date_key),Time filter on relationships
6,dw_dim_customer,DISTKEY(customer_id),Collocate NK during ETL lookups
7,dw_dim_product,DISTKEY(product_id),Collocate NK during ETL lookups
8,dw_dim_date and junk dims,DISTSTYLE ALL,Broadcast; tables are tiny


Creating materialized view dw_mv_daily_revenue
MV created. After future loads run: REFRESH MATERIALIZED VIEW public.dw_mv_daily_revenue;


In [28]:
print("Running ANALYZE on warehouse tables")
for table in [
    "dw_fact_orders",
    "dw_fact_events",
    "dw_fact_graph_edges",
    "dw_dim_customer",
    "dw_dim_product",
    "dw_dim_date",
]:
    rs_exec("ANALYZE public." + table + ";")
    print("  ANALYZE", table)

info_sql = (
    "SELECT table_schema, table_name, table_type "
    "FROM information_schema.tables "
    "WHERE table_schema = 'public' "
    "AND (table_name LIKE 'dw_%' OR table_name LIKE 'stg_%') "
    "ORDER BY table_name"
)
info = rs_exec(info_sql, return_results=True)
display(pd.DataFrame(info))

print("Task 5 complete")

Running ANALYZE on warehouse tables
  ANALYZE dw_fact_orders
  ANALYZE dw_fact_events
  ANALYZE dw_fact_graph_edges
  ANALYZE dw_dim_customer
  ANALYZE dw_dim_product
  ANALYZE dw_dim_date


,table_schema,table_name,table_type
0,public,dw_dim_ab_variant,BASE TABLE
1,public,dw_dim_browser,BASE TABLE
2,public,dw_dim_campaign,BASE TABLE
3,public,dw_dim_channel,BASE TABLE
4,public,dw_dim_customer,BASE TABLE
5,public,dw_dim_date,BASE TABLE
6,public,dw_dim_device,BASE TABLE
7,public,dw_dim_os,BASE TABLE
8,public,dw_dim_payment_method,BASE TABLE
9,public,dw_dim_product,BASE TABLE


Task 5 complete


---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [29]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)
row_counts = rs_exec(
    "SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS n FROM public.stg_orders_raw "
    "UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw "
    "UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw "
    "UNION ALL SELECT 'dw_dim_date', COUNT(*) FROM public.dw_dim_date "
    "UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer "
    "UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product "
    "UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders "
    "UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events "
    "UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges",
    return_results=True,
)
print("Row counts (staging vs facts should match for the three facts)")
display(pd.DataFrame(row_counts))

quality = rs_exec(
    "SELECT 'fact_orders null customer_sk' AS check_name, COUNT(*) AS n FROM public.dw_fact_orders WHERE customer_sk IS NULL "
    "UNION ALL SELECT 'fact_orders duplicate order_id', COUNT(*) - COUNT(DISTINCT order_id) FROM public.dw_fact_orders "
    "UNION ALL SELECT 'fact_events null customer_sk', COUNT(*) FROM public.dw_fact_events WHERE customer_sk IS NULL "
    "UNION ALL SELECT 'fact_orders negative totals', COUNT(*) FROM public.dw_fact_orders WHERE order_total_usd < 0 "
    "UNION ALL SELECT 'stg minus fact orders', (SELECT COUNT(*) FROM public.stg_orders_raw)-(SELECT COUNT(*) FROM public.dw_fact_orders) "
    "UNION ALL SELECT 'stg minus fact events', (SELECT COUNT(*) FROM public.stg_events_raw)-(SELECT COUNT(*) FROM public.dw_fact_events) "
    "UNION ALL SELECT 'stg minus fact edges', (SELECT COUNT(*) FROM public.stg_edges_raw)-(SELECT COUNT(*) FROM public.dw_fact_graph_edges)",
    return_results=True,
)
print("Quality checks (expect zeros)")
display(pd.DataFrame(quality))


TASK 6: Validation and Reporting
Row counts (staging vs facts should match for the three facts)


,table_name,n
0,dw_fact_graph_edges,2500
1,dw_fact_events,2500
2,stg_edges_raw,2500
3,dw_dim_date,552
4,dw_dim_customer,6825
5,stg_orders_raw,2500
6,dw_fact_orders,2500
7,stg_events_raw,2500
8,dw_dim_product,3360


Quality checks (expect zeros)


,check_name,n
0,fact_orders negative totals,0
1,stg minus fact events,0
2,stg minus fact edges,0
3,fact_orders null customer_sk,0
4,fact_orders duplicate order_id,0
5,fact_events null customer_sk,0
6,stg minus fact orders,0


In [30]:
print("Sample analytics - daily revenue from MV")
daily_rev = rs_exec(
    "SELECT d.date_actual, mv.orders, mv.revenue_usd, mv.avg_order_value, mv.on_time_rate, mv.returned_orders "
    "FROM public.dw_mv_daily_revenue mv "
    "JOIN public.dw_dim_date d ON d.date_key = mv.order_date_key "
    "ORDER BY d.date_actual LIMIT 15",
    return_results=True,
)
display(pd.DataFrame(daily_rev))

print("Clickstream funnel")
funnel = rs_exec(
    "SELECT event_type, COUNT(*) AS events, COUNT(DISTINCT customer_sk) AS customers "
    "FROM public.dw_fact_events GROUP BY event_type ORDER BY events DESC",
    return_results=True,
)
display(pd.DataFrame(funnel))

print("Graph relationship mix")
rels = rs_exec(
    "SELECT relationship, COUNT(*) AS n FROM public.dw_fact_graph_edges "
    "GROUP BY relationship ORDER BY n DESC",
    return_results=True,
)
display(pd.DataFrame(rels))


Sample analytics - daily revenue from MV


,date_actual,orders,revenue_usd,avg_order_value,on_time_rate,returned_orders
0,2024-01-01,6,4086.97,681.16,0.8,2
1,2024-01-02,5,2150.01,430.00,0.6,1
2,2024-01-03,6,681.74,113.62,0.8,1
3,2024-01-04,3,731.67,243.89,1.0,0
4,2024-01-05,4,1302.17,325.54,1.0,0
5,2024-01-06,9,4421.65,491.29,0.6,1
6,2024-01-07,5,2293.13,458.62,0.8,1
7,2024-01-09,6,1712.60,285.43,0.8,1
8,2024-01-10,8,2472.00,309.00,1.0,2
9,2024-01-11,6,4127.43,687.90,0.8,2


Clickstream funnel


,event_type,events,customers
0,product_view,696,693
1,page_view,641,636
2,add_to_cart,480,479
3,checkout_start,259,259
4,payment_attempt,218,218
5,purchase,155,155
6,return_initiated,51,51


Graph relationship mix


,relationship,n
0,VIEWED,830
1,PURCHASED,469
2,ADDED_TO_CART,441
3,ALSO_BOUGHT_WITH,393
4,REFERRED_FRIEND,256
5,RETURNS,111


In [31]:
print("Generating Final Report")
report_content = (
    "# Data Warehouse Build Report\n\n"
    "Generated: " + datetime.utcnow().isoformat() + "Z\n\n"
    "## Schema overview\n"
    "Star schema matching project-ddl-long.md and project-mermaid-diagram.md.\n\n"
    "Staging: stg_orders_raw, stg_events_raw, stg_edges_raw\n"
    "Facts: dw_fact_orders (order grain), dw_fact_events (event grain), dw_fact_graph_edges (edge grain)\n"
    "Dimensions: date, customer (SCD2-ready), product (SCD2-ready), plus junk dims.\n\n"
    "## Design rationale\n"
    "1. Star schema so analysts do not join Postgres + Cassandra + Neo4j at query time.\n"
    "2. Surrogate keys isolate facts from changing customer segment / product category.\n"
    "3. DISTKEY(customer_sk) on orders and events collocates revenue + funnel.\n"
    "4. DISTKEY(to_product_sk) on graph edges supports also-bought analysis.\n"
    "5. SORTKEY date keys prune time filters. Small dims DISTSTYLE ALL.\n"
    "6. MV dw_mv_daily_revenue stores daily GMV, AOV, on-time rate, returns.\n\n"
    "## Refresh\n"
    "After each load: REFRESH MATERIALIZED VIEW public.dw_mv_daily_revenue; ANALYZE key tables.\n\n"
    "## Analytics supported\n"
    "Daily GMV, delivery SLA, return rate, clickstream funnel, A/B, recommendation-graph mix.\n"
)
report_path = os.path.join(BASE_DIR, "warehouse_report.md")
with open(report_path, "w") as f:
    f.write(report_content)
print("Report saved to", report_path)
print("="*60)
print("ALL TASKS COMPLETE")
print("="*60)


Generating Final Report
Report saved to ./warehouse_report.md
ALL TASKS COMPLETE
